# A06 · 17×17: failure modes

The two typical ways training fails, for the same two runs as in A05. We plot the **death rate**, the **game
length** and the **bombs per game** during training. Very short games mean the agent kills itself early.
Games that always last the full $200$ steps without bombs mean the agent has become passive. The second
figure shows how the evaluation games against the rule-based agent end: the agent killed itself, was killed
by the opponent, survived but lost, or survived and won.

**Reads** `runs/a2c_17x17/` and `runs/a2c_17x17_03/` (T03). **Writes** `figures/a2c_17x17_failure_modes.pdf`
and `figures/a2c_17x17_outcomes.pdf`.

In [ ]:
import json
import pathlib
import shutil

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns

HERE = pathlib.Path.cwd().resolve()  # run from the notebook's folder or from the repository root
EXP = next(p / "experiments" for p in (HERE, *HERE.parents) if (p / "experiments" / "README.md").exists())
RUNS = EXP / "runs"
FIGURES = EXP / "figures"
FIGURES.mkdir(exist_ok=True)

TEXTWIDTH = 16.0 / 2.54
sns.set_theme(context="paper", style="ticks", palette="colorblind", font="serif", rc={
    "text.usetex": shutil.which("latex") is not None,
    "text.latex.preamble": r"\usepackage[T1]{fontenc}\usepackage{amsmath}",
    "font.serif": ["cmr10", "Computer Modern Roman"], "mathtext.fontset": "cm", "axes.formatter.use_mathtext": True,
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9, "legend.fontsize": 8, "legend.title_fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.grid": True, "grid.linewidth": 0.4, "grid.alpha": 0.6,
    "lines.linewidth": 1.1, "figure.dpi": 120, "figure.constrained_layout.use": True,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "pdf.fonttype": 42,
})

## Data

Both runs are identical except for the crate reward in the crate stage.

In [ ]:
RUN_NAMES = {"crate bonus 0.6": "a2c_17x17", "crate bonus 0.3": "a2c_17x17_03"}  # label: run folder
COLORS = {"crate bonus 0.6": "C0", "crate bonus 0.3": "C3"}
SMOOTH = 25  # rolling mean over iterations, restarted at every stage

available = {label: run for label, run in RUN_NAMES.items() if (RUNS / run / "history.csv").exists()}
config = json.loads((RUNS / next(iter(available.values())) / "config.json").read_text())
history = pd.concat([pd.read_csv(RUNS / run / "history.csv").query("network == 'FC128'").assign(run=label)
                     for label, run in available.items()], ignore_index=True).sort_values("iteration")
LABELS = list(available)
boundaries = pd.Series([s["iterations"] for s in config["stages"]]).cumsum()
stage_starts = [0, *boundaries[:-1]]


def smoothed(columns):
    frame = history.copy()
    for column in columns:
        frame[column] = frame.groupby(["run", "stage"])[column].transform(
            lambda s: s.rolling(SMOOTH, min_periods=1).mean())
    return frame


def stage_axes(axes):
    """Mark where each stage starts with a dashed line and label the stages on top."""
    for ax in axes:
        for x in boundaries[:-1]:
            ax.axvline(x, color="0.3", ls="--", lw=0.8)
    axes[-1].set_xlabel("training iteration (cumulative over stages)")
    axes[-1].set_xlim(0, boundaries.iloc[-1])
    top = axes[0].secondary_xaxis("top")
    top.set_xticks([start + stage["iterations"] / 2 for start, stage in zip(stage_starts, config["stages"])],
                   [stage["name"] for stage in config["stages"]])
    top.tick_params(length=0, pad=2)


def lines(frame, axes, metrics):
    for ax, (column, label) in zip(axes, metrics.items()):
        sns.lineplot(frame, x="iteration", y=column, hue="run", hue_order=LABELS, palette=COLORS, ax=ax,
                     legend=False, lw=1.0)
        ax.set_ylabel(label)
    handles = [plt.Line2D([], [], color=COLORS[label], lw=1.2) for label in LABELS]
    axes[0].figure.legend(handles, LABELS, loc="outside upper center", ncol=len(LABELS), frameon=False)

history.groupby(["run", "stage"]).size().unstack()

## Failure modes over training

In [ ]:
METRICS = {"death": "death rate", "world_steps": "episode\nlength", "bombs": "bombs per\nepisode"}
fig, axes = plt.subplots(len(METRICS), 1, sharex=True, figsize=(TEXTWIDTH, 0.6 * TEXTWIDTH))
lines(smoothed(METRICS), axes, METRICS)
axes[0].set_ylim(0, 1)
axes[1].axhline(config["stages"][0]["rules"]["max_steps"], color="0.5", ls=":", lw=0.8)  # the step limit
stage_axes(axes)
fig.align_ylabels(axes)
fig.savefig(FIGURES / "a2c_17x17_failure_modes.pdf")

## Outcomes of the evaluation games

Every evaluation game ends in one of four ways. A tie counts as a loss.

In [ ]:
OUTCOMES = ["self-kill", "killed by rival", "survived, lost", "survived, won"]
games = pd.concat([pd.read_csv(RUNS / run / "eval.csv").assign(run=label) for label, run in available.items()],
                  ignore_index=True)
games["outcome"] = np.select(
    [games.self_kill == 1, games.death == 1, games.win == 0], OUTCOMES[:3], OUTCOMES[3])
shares = (games.groupby("run").outcome.value_counts(normalize=True).unstack(fill_value=0.0)
          .reindex(index=LABELS[::-1], columns=OUTCOMES, fill_value=0.0))

fig, ax = plt.subplots(figsize=(TEXTWIDTH, 0.12 * TEXTWIDTH + 0.09 * TEXTWIDTH * len(shares)))
left = np.zeros(len(shares))
percent = r"\%" if plt.rcParams["text.usetex"] else "%"
for outcome, color in zip(OUTCOMES, ["C3", "C1", "0.65", "C2"]):
    ax.barh(shares.index, shares[outcome], left=left, color=color, label=outcome, height=0.6)
    for y, (x, w) in enumerate(zip(left, shares[outcome])):
        if w >= 0.06:
            ax.text(x + w / 2, y, f"{w:.0%}".replace("%", percent), ha="center", va="center", fontsize=7,
                    color="black" if color == "0.65" else "white")
    left += shares[outcome].to_numpy()
ax.set(xlim=(0, 1), xlabel=f"share of {len(games) // len(shares)} evaluation games vs. rule-based agent")
ax.grid(False, axis="y")
ax.legend(ncol=4, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0))
fig.savefig(FIGURES / "a2c_17x17_outcomes.pdf")
games.groupby("run")[["score", "rival_score", "win", "coins", "death", "self_kill", "steps"]].mean().round(3)